Scott Lepley, Age of Learning Case, May 2026

In [1]:
import pandas as pd
import duckdb
import plotly.express as px
from scipy import stats
from scipy.stats import chi2_contingency
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.linear_model import LogisticRegression
pd.set_option("display.max_rows", 50)

In [2]:
#i converted the sent file from xlsx to csv
df = pd.read_csv("AoLAnalyticsTask.csv")

In [3]:
print("SHAPE")
print(df.shape)
print("\n\nCOLUMNS")
print(df.dtypes.to_string())

SHAPE
(7706, 37)


COLUMNS
parentid                     int64
postal_code                float64
subd                           str
covid                          str
free_trial                    bool
payment_frequency_group        str
userid                       int64
gender                         str
path_level_grade               str
age                        float64
which_day                      str
n_act                      float64
n_area                     float64
n_session                  float64
n_completes                float64
n_clicks                   float64
tot_ts_mins                float64
shopped                    float64
device                         str
art                        float64
books                      float64
game                       float64
learn_music                float64
puzzles                    float64
song_chants                float64
abcmtv                     float64
about_me                   float64
aquarium_hamster           f

In [4]:
#null values
null_counts = df.isnull().sum().reset_index()
null_counts.columns = ["column", "null_count"]
null_counts

,column,null_count
0,parentid,0
1,postal_code,178
2,subd,0
3,covid,0
4,free_trial,0
5,payment_frequency_group,0
6,userid,0
7,gender,97
8,path_level_grade,0
9,age,234


In [5]:
#duckdb sql querying
con = duckdb.connect()

con.execute("SELECT * FROM 'AoLAnalyticsTask.csv' LIMIT 10").df()

,parentid,postal_code,subd,covid,free_trial,payment_frequency_group,userid,gender,path_level_grade,age,...,aquarium_hamster,clsrm_subj,learn_at_home,library,mastery_math,mastery_reading,nav,pet_park,shopping,zoo_farm_theater
0,4140783856,NA,Wed,after,True,Monthly,4140862206,M,PreK,5,...,2.41605,0,0,0,16.37438333,0,1.848866667,9.810566667,0.673116667,0
1,4140783856,NA,Wed,after,True,Monthly,4140862206,M,PreK,5,...,0,0,0,0,0,0,0.2924,0,0,0
2,4177341908,NA,Thu,after,False,Annual,4177470108,F,First Grade,7,...,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA
3,4177341908,NA,Thu,after,False,Annual,4177470108,F,First Grade,7,...,5.111383333,0.270383333,0,0.567483333,0,0,11.38418333,0.831116667,0,0
4,4139574706,NA,Wed,after,True,Monthly,4140210706,M,Kindergarten,6,...,14.21661667,0.909283333,0,0,0,0,8.3761,12.64668333,0,1.511416667
5,4139574706,NA,Wed,after,True,Monthly,4140210706,M,Kindergarten,6,...,18.08923333,0,0,0,0,0,2.240516667,1.39855,0,0
6,4225269558,NA,Mon,after,True,Monthly,4225448308,F,PreK,5,...,0,0,0,0,0,0,0.906316667,0,7.7228,0
7,4225269558,NA,Mon,after,True,Monthly,4225448308,F,PreK,5,...,18.70446667,0,0,0,9.380683333,0,6.9553,0,8.2508,0
8,4254072958,NA,Tue,after,True,Monthly,4254093558,F,Preschool,5,...,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA
9,4254072958,NA,Tue,after,True,Monthly,4254093558,F,Preschool,5,...,1.3569,0,0,0.763283333,0.817383333,0,4.86435,0,0,0


The above was exploratory, and we add visuals etc to be icing on the cake. For now, i'm going to dig into the question

# Question 1: Is there a significant difference in what users do on day0 compared to day1?

Data dictionary thing to be aware of: activities completed is sort of ambiguous

N_completes: number of completed activities (art, books, games, learn_music, puzzles)\
n_act: number of activities completed (e.g., read a book, played a game)

First, I'll answer what percentage of our users in the dataset make it to day 1

In [6]:

con.execute("""
    SELECT
        COUNT(*) AS total_day0_users,
        SUM(retained) AS retained_users,
        ROUND(SUM(retained) * 100.0 / COUNT(*), 2) AS retention_rate
    FROM (
        SELECT
            d0.userid,
            CASE WHEN d1.userid IS NOT NULL THEN 1 ELSE 0 END AS retained
        FROM 'AoLAnalyticsTask.csv' d0
        LEFT JOIN 'AoLAnalyticsTask.csv' d1
            ON d0.userid = d1.userid
            AND d1.which_day = 'day1'
            AND d1.n_act IS NOT NULL
        WHERE d0.which_day = 'day0'
    )
""").df()

,total_day0_users,retained_users,retention_rate
0,4358,2706.0,62.09


Our total retention rate is 62%

To think about this question:

First, I'd look at aggregate day0 behavior vs aggregate day1 behavior. This might be what the default dashboard shows. It's helpful as context but misleading without the caveat that day1 users are a self-selected retained group, who are by definition more engaged. So the differences you might see between day0 and day1 aggregate values could be explained entirely by the fact that you're looking at a more engaged cohort of people on day1

Next, I'm going to oberve only retained users, a paired comparison. I only include day0 users who also appear on day1

And last, I'm going to look at churned users. day0 behavbior between those came back and those who didn't

In [7]:
all_engagement_df = con.execute("""
    SELECT
        which_day,
        ROUND(AVG(TRY_CAST(tot_ts_mins AS DOUBLE)), 2) AS avg_time_mins,
        ROUND(AVG(TRY_CAST(n_completes AS DOUBLE)), 2) AS avg_completes,
        ROUND(AVG(TRY_CAST(n_act AS DOUBLE)), 2) AS avg_activities,
        ROUND(AVG(TRY_CAST(n_session AS DOUBLE)), 2) AS avg_sessions,
        ROUND(AVG(TRY_CAST(n_area AS DOUBLE)), 2) AS avg_areas,
        ROUND(AVG(TRY_CAST(n_clicks AS DOUBLE)), 2) AS avg_clicks,
        COUNT(*) AS total_users
    FROM 'AoLAnalyticsTask.csv'
    WHERE which_day IN ('day0', 'day1')
    AND n_act IS NOT NULL
    GROUP BY which_day
""").df()

In [8]:
all_engagement_df

,which_day,avg_time_mins,avg_completes,avg_activities,avg_sessions,avg_areas,avg_clicks,total_users
0,day1,78.71,15.53,4.73,3.81,7.56,80.36,2979
1,day0,64.64,14.14,4.66,2.86,7.44,68.71,4358


Query 1 shows day1 looking more engaged than day0, but again, that's likely misleading. Day1 retained users are a self-selected cohort of people who already had higher enthusiasm for the platform. Query 2 below controls for this by comparing the same users across both days

below i'm going to look at a subsection of users where we only consider users who were retained. so day0 stats will reflect users who also appeared on day1

In [9]:
engagement_df_rt = con.execute("""
    WITH retained_users AS (
        SELECT DISTINCT userid
        FROM 'AoLAnalyticsTask.csv'
        WHERE which_day = 'day1'
        AND n_act IS NOT NULL
    )
    SELECT
        which_day,
        ROUND(AVG(TRY_CAST(tot_ts_mins AS DOUBLE)), 2) AS avg_time_mins,
        ROUND(AVG(TRY_CAST(n_completes AS DOUBLE)), 2) AS avg_completes,
        ROUND(AVG(TRY_CAST(n_act AS DOUBLE)), 2) AS avg_activities,
        ROUND(AVG(TRY_CAST(n_session AS DOUBLE)), 2) AS avg_sessions,
        ROUND(AVG(TRY_CAST(n_area AS DOUBLE)), 2) AS avg_areas,
        ROUND(AVG(TRY_CAST(n_clicks AS DOUBLE)), 2) AS avg_clicks,
        COUNT(*) AS total_users
    FROM 'AoLAnalyticsTask.csv'
    WHERE which_day IN ('day0', 'day1')
    AND n_act IS NOT NULL
    AND userid IN (SELECT userid FROM retained_users)
    GROUP BY which_day
""").df()

In [10]:
engagement_df_rt

,which_day,avg_time_mins,avg_completes,avg_activities,avg_sessions,avg_areas,avg_clicks,total_users
0,day1,78.71,15.53,4.73,3.81,7.56,80.36,2979
1,day0,79.79,17.65,5.03,3.29,8.08,84.22,2706


In [11]:
# melt the dataframe for plotting
plot_df = engagement_df_rt.melt(
    id_vars='which_day',
    value_vars=['avg_time_mins', 'avg_completes', 'avg_activities', 'avg_sessions', 'avg_areas', 'avg_clicks'],
    var_name='metric',
    value_name='value'
)

fig = px.bar(
    plot_df,
    x='metric',
    y='value',
    color='which_day',
    barmode='group',
    title='Retained Users: Day0 vs Day1 Engagement',
    labels={'metric': 'Metric', 'value': 'Average', 'which_day': 'Day'}
)

fig.update_layout(
    plot_bgcolor='white',
    yaxis=dict(gridcolor='lightgrey')
)

fig.show()

Looking at these results above, we see the selection bias now visible. Our first query was self selecting for users more interested in the product. Our 2nd query (above), shows that retained users were already highly engaged on day0. The product held their attention but they actually engaged slightly less deeply in some ways on their second visit. Maybe they were exploring more on their initial play session. 

Below - churned vs retained users

In [12]:
churn_df = con.execute("""
    WITH retained_users AS (
        SELECT DISTINCT userid
        FROM 'AoLAnalyticsTask.csv'
        WHERE which_day = 'day1'
        AND n_act IS NOT NULL
    )
    SELECT
        CASE WHEN r.userid IS NOT NULL THEN 'retained' ELSE 'churned' END AS user_type,
        ROUND(AVG(TRY_CAST(tot_ts_mins AS DOUBLE)), 2) AS avg_time_mins,
        ROUND(AVG(TRY_CAST(n_completes AS DOUBLE)), 2) AS avg_completes,
        ROUND(AVG(TRY_CAST(n_act AS DOUBLE)), 2) AS avg_activities,
        ROUND(AVG(TRY_CAST(n_session AS DOUBLE)), 2) AS avg_sessions,
        ROUND(AVG(TRY_CAST(n_area AS DOUBLE)), 2) AS avg_areas,
        ROUND(AVG(TRY_CAST(n_clicks AS DOUBLE)), 2) AS avg_clicks,
        COUNT(*) AS total_users
    FROM 'AoLAnalyticsTask.csv' a
    LEFT JOIN retained_users r ON a.userid = r.userid
    WHERE a.which_day = 'day0'
    AND a.n_act IS NOT NULL
    GROUP BY user_type
""").df()

In [13]:
churn_df

,user_type,avg_time_mins,avg_completes,avg_activities,avg_sessions,avg_areas,avg_clicks,total_users
0,retained,79.79,17.65,5.03,3.29,8.08,84.22,2706
1,churned,39.81,8.40,4.05,2.14,6.40,43.32,1652


Query 3 shows us retained users were dramatically more engaged on day0 than users who churned, across every metric, which is intuitive. They spent twice as long in the app, completed twice as many activities, had more sessions, and clicked more. 

# What's all this mean?

Among retained users, behavior isn't that different from day0 to day1. There's small statistical differences, but effect sizes are negligible and ultimately it doesn't mean much. Sessions increased slightly, but everything else held steady or was slighlty lower but not meaningfully so.

I think the more interesting finding is exploring the differences between churned and retained users on day0. Retained users were twice as engaged on day0.

It means what we're looking for is what distinguished users who came back on d1 vs those who didn't. Retained users were roughly twice as engaged on their first day. This suggests day0 engagement is a strong predictor of retention. We want to know 'what are retained users finding valuable that churned users aren't?'

## **So, is there a significant difference in what users do on day0 compared to day1?**
The short answer is no. Among retained users, behavior on day0 vs day1 is not meaningfully different. There are small statistical differences but effect sizes are negligible

As a follow up, we could cointnue this by digging into whether certain user segments retain at higher rates. This could help product prioritize which users to focus on for improving day0 engagement. 

In [14]:
sql_df = con.execute("""
    WITH base AS (
        SELECT
            d0.userid,
            d0.path_level_grade,
            d0.device,
            d0.subd,
            d0.free_trial,
            d0.payment_frequency_group,
            CASE WHEN d1.userid IS NOT NULL THEN 1 ELSE 0 END AS retained
        FROM 'AoLAnalyticsTask.csv' d0
        LEFT JOIN 'AoLAnalyticsTask.csv' d1
            ON d0.userid = d1.userid
            AND d1.which_day = 'day1'
            AND d1.n_act IS NOT NULL
        WHERE d0.which_day = 'day0'
    )
    SELECT 'path_level_grade' AS segment, path_level_grade AS segment_value, COUNT(*) AS total, SUM(retained) AS retained, ROUND(SUM(retained) * 100.0 / COUNT(*), 2) AS retention_rate FROM base GROUP BY path_level_grade
    UNION ALL
    SELECT 'device', device, COUNT(*), SUM(retained), ROUND(SUM(retained) * 100.0 / COUNT(*), 2) FROM base GROUP BY device
    UNION ALL
    SELECT 'subd', subd, COUNT(*), SUM(retained), ROUND(SUM(retained) * 100.0 / COUNT(*), 2) FROM base GROUP BY subd
    UNION ALL
    SELECT 'free_trial', free_trial, COUNT(*), SUM(retained), ROUND(SUM(retained) * 100.0 / COUNT(*), 2) FROM base GROUP BY free_trial
    UNION ALL
    SELECT 'payment_frequency_group', payment_frequency_group, COUNT(*), SUM(retained), ROUND(SUM(retained) * 100.0 / COUNT(*), 2) FROM base GROUP BY payment_frequency_group
    ORDER BY segment, retention_rate DESC
""").df()

In [15]:
sql_df

,segment,segment_value,total,retained,retention_rate
0,device,ipad,1267,902.0,71.19
1,device,android-tab,713,453.0,63.53
2,device,other,278,165.0,59.35
3,device,android-phone,394,232.0,58.88
4,device,Windows,731,411.0,56.22
5,device,Mac OS X,330,185.0,56.06
6,device,iphone,645,358.0,55.50
7,free_trial,false,677,468.0,69.13
8,free_trial,true,3681,2238.0,60.80
9,path_level_grade,Kindergarten,1011,709.0,70.13


In [16]:
# filter to payment frequency segment
payment_df = sql_df[sql_df['segment'] == 'payment_frequency_group'].sort_values('retention_rate', ascending=False)

fig = px.bar(
    payment_df,
    x='segment_value',
    y='retention_rate',
    text='retention_rate',
    custom_data=['total', 'retained'],
    title='Day 0 → Day 1 Retention Rate by Payment Frequency',
    labels={'segment_value': 'Payment Frequency', 'retention_rate': 'Retention Rate (%)'}
)

fig.update_traces(
    texttemplate='%{text}%',
    textposition='outside',
    hovertemplate='<b>%{x}</b><br>Retention Rate: %{y}%<br>Total Users: %{customdata[0]}<br>Retained: %{customdata[1]}<extra></extra>'
)

fig.update_layout(
    yaxis_range=[0, 100],
    plot_bgcolor='white',
    yaxis=dict(gridcolor='lightgrey')
)

fig.show()

Ultimately, our findings from above show that retention signals we are using are generally a proxy for underlying user excitement. The hard product questoin is 'how do we create genuine excitement on day0 for users who don't already have it?'

### Question 2: Our Product team recently added new content to our ABCmouse app that features academic-type activities focused on math skills. Their hypothesis is that this content will appeal more to parents and children who use the app as a part of school-related learning as opposed to those who use the app as more of a recreational game. Our Product stakeholder tells you that they believe that recreational users are more likely to do “Pet_Park” activities, which involve playing with and decorating virtual pets, while academic users are more likely to do “Mastery_Math” activities. Does the data show that the users who spend time in “Pet_Park” are different than those who spend time in “Mastery_Math”? Please provide evidence to support your answer.


This is a user segmentation/clustering problem. My instincts say to do the following:
First: Challenge the assumption. Product believes in different behavioral patterns between these two segments, recreational and academic users. I can test whether or not this assumption holds in the data by plotting the distribution of usage of pet park and mastery math and looking for bimodality. If I see a normal distribution, the premise of this hypothesis is wrong and the common usage pattern is to use both equally. If I see bimodality, that confirms we have two distinct usage patterns and the PMs intuition is supported.

Next: Answer their question. 
Are the users in pet park different from mastery_math users? 

Lastly: Propose more robust segmentations

Things I considered as input for segmentation: 
- a = time spent in mastery_math
- b = time spent in pet_park
- a + b = total engagement in both of these content areas
- a/b = ratio

First, I'll calculate each user's time in pet_park and mastery_math. I'll also define the ratio as pet_park / (pet_park + matery_math)
A value of 0 means all mastery_math, a value of 1 means all pet_park, 
and 0.5 means equal time in both. I'm filtering to users who have 
spent time in at least one of the two content areas.


In [17]:
dist_df = con.execute("""
    WITH user_totals AS (
        SELECT
            userid,
            SUM(TRY_CAST(pet_park AS DOUBLE)) AS total_pet_park,
            SUM(TRY_CAST(mastery_math AS DOUBLE)) AS total_mastery_math
        FROM 'AoLAnalyticsTask.csv'
        GROUP BY userid
    )
    SELECT
        userid,
        total_pet_park,
        total_mastery_math,
        CASE 
            WHEN (total_pet_park + total_mastery_math) > 0 
            THEN total_pet_park / (total_pet_park + total_mastery_math)
            ELSE NULL 
        END AS pet_park_ratio
    FROM user_totals
    WHERE total_pet_park > 0 OR total_mastery_math > 0
""").df()

Also to keep note of - how many users have no time in either pet park or math mastery?

In [18]:
con.execute("""
    WITH user_totals AS (
        SELECT
            userid,
            SUM(TRY_CAST(pet_park AS DOUBLE)) AS total_pet_park,
            SUM(TRY_CAST(mastery_math AS DOUBLE)) AS total_mastery_math
        FROM 'AoLAnalyticsTask.csv'
        GROUP BY userid
    )
    SELECT
        COUNT(*) AS total_users,
        SUM(CASE WHEN total_pet_park = 0 AND total_mastery_math = 0 THEN 1 ELSE 0 END) AS neither_users,
        ROUND(SUM(CASE WHEN total_pet_park = 0 AND total_mastery_math = 0 THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 1) AS pct_neither
    FROM user_totals
""").df()

,total_users,neither_users,pct_neither
0,5000,1733.0,34.7


1733 users, or 34.7% of total, never engage with either pet_park or mastery_math. These users are outside of the scope of the question asked so they're excluded from analysis. 


Next, I'll plot the distribution of the pet park ratio to test for bimodality. 
If there are two distinct user types i'd expect to see spikes at both ends of the distribution

In [19]:
fig = px.histogram(
    dist_df,
    x='pet_park_ratio',
    nbins=20,
    title='Distribution of Pet Park vs Mastery Math Time Share',
    labels={'pet_park_ratio': 'Pet Park Share (0=all Mastery Math, 1=all Pet Park)'}
)

fig.update_layout(
    plot_bgcolor='white',
    yaxis=dict(gridcolor='lightgrey')

)

fig.show()

Above chart shows us spikes at 0 and 1, clear evidence of some bimodality. But, the middle is not empty by any means. The chart tells us two things:
1. Two distinct extreme user types exist and support the PM's hypothesis
2. Most users don't fit cleanly into either bucket and are mixed usage

In [20]:
segment_counts = con.execute("""
    WITH user_totals AS (
        SELECT
            userid,
            SUM(TRY_CAST(pet_park AS DOUBLE)) AS total_pet_park,
            SUM(TRY_CAST(mastery_math AS DOUBLE)) AS total_mastery_math
        FROM 'AoLAnalyticsTask.csv'
        GROUP BY userid
    ),
    ratios AS (
        SELECT
            userid,
            total_pet_park,
            total_mastery_math,
            CASE 
                WHEN (total_pet_park + total_mastery_math) > 0 
                THEN total_pet_park / (total_pet_park + total_mastery_math)
                ELSE NULL 
            END AS pet_park_ratio
        FROM user_totals
        WHERE total_pet_park > 0 OR total_mastery_math > 0
    )
    SELECT
        CASE
            WHEN pet_park_ratio = 0 THEN 'Pure Academic (ratio = 0)'
            WHEN pet_park_ratio < 0.1 THEN 'Mostly Academic (0 < ratio < 0.1)'
            WHEN pet_park_ratio <= 0.9 THEN 'Mixed (0.1 to 0.9)'
            WHEN pet_park_ratio < 1 THEN 'Mostly Recreational (0.9 < ratio < 1)'
            WHEN pet_park_ratio = 1 THEN 'Pure Recreational (ratio = 1)'
        END AS user_type,
        COUNT(*) AS total_users,
        ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 1) AS pct_of_users
    FROM ratios
    GROUP BY user_type
    ORDER BY MIN(pet_park_ratio)
""").df()

In [21]:
segment_counts

,user_type,total_users,pct_of_users
0,Pure Academic (ratio = 0),982,34.7
1,Mostly Academic (0 < ratio < 0.1),204,7.2
2,Mixed (0.1 to 0.9),822,29.0
3,Mostly Recreational (0.9 < ratio < 1),109,3.8
4,Pure Recreational (ratio = 1),715,25.2


In [22]:
fig = px.bar(
    segment_counts,
    x='user_type',
    y='total_users',
    text='pct_of_users',
    title='User Distribution by Content Preference',
    labels={'user_type': 'User Type', 'total_users': 'Number of Users'}
)

fig.update_traces(
    texttemplate='%{text}%',
    textposition='outside'
)

fig.update_layout(
    plot_bgcolor='white',
    yaxis=dict(gridcolor='lightgrey'),
    yaxis_range=[0, 1200]
)

fig.show()

We do see that 71% of users fall into either "Pure Academic", "Pure Recreational", "Mostly Recreational" or "Mostly Academic". This supports the PMs hypothesis. 
From this, we could reasonably propose the following segments: Academic, Recreational, and Mixed

In [23]:
dist_df = con.execute("""
    WITH user_totals AS (
        SELECT
            userid,
            MAX(path_level_grade) AS path_level_grade,
            MAX(free_trial) AS free_trial,
            MAX(payment_frequency_group) AS payment_frequency_group,
            MAX(device) AS device,
            MAX(gender) as gender,
            MAX(covid) as covid,
            MAX(device) AS device,
            AVG(TRY_CAST(age AS DOUBLE)) AS avg_age,
            SUM(TRY_CAST(tot_ts_mins AS DOUBLE)) AS total_time,
            SUM(TRY_CAST(n_completes AS DOUBLE)) AS total_completes,
            SUM(TRY_CAST(n_act AS DOUBLE)) AS total_activities,
            SUM(TRY_CAST(pet_park AS DOUBLE)) AS total_pet_park,
            SUM(TRY_CAST(mastery_math AS DOUBLE)) AS total_mastery_math,
            MAX(CASE WHEN which_day = 'day1' AND n_act IS NOT NULL THEN 1 ELSE 0 END) AS retained
        FROM 'AoLAnalyticsTask.csv'
        GROUP BY userid
        HAVING SUM(TRY_CAST(pet_park AS DOUBLE)) > 0
            OR SUM(TRY_CAST(mastery_math AS DOUBLE)) > 0
    )
    SELECT *,
        total_pet_park / NULLIF(total_pet_park + total_mastery_math, 0) AS pet_park_ratio
    FROM user_totals
""").df()

# reapply segmentation
dist_df['total_combined'] = dist_df['total_mastery_math'] + dist_df['total_pet_park']
dist_df['segment'] = None
mask = dist_df['total_combined'] >= 5
dist_df.loc[mask & (dist_df['pet_park_ratio'] <= 0.1), 'segment'] = 'Academic'
dist_df.loc[mask & (dist_df['pet_park_ratio'] > 0.1) & (dist_df['pet_park_ratio'] < 0.9), 'segment'] = 'Mixed'
dist_df.loc[mask & (dist_df['pet_park_ratio'] >= 0.9), 'segment'] = 'Recreational'
dist_df['segment'] = dist_df['segment'].fillna('Disengaged')

print(dist_df['segment'].value_counts())

segment
Disengaged      902
Academic        752
Mixed           727
Recreational    451
Name: count, dtype: int64


Added a flag for "Disengaged" users - which is users who touched either content area, but spent less than 5 minutes and didn't engage meaningfully with either. There's potentially a product opportunity here for these people. 

Next, I'll segment these users

In [24]:
# engagement profile
print("=== Engagement by Segment ===")
print(dist_df.groupby('segment')[['total_time', 'total_completes', 'total_activities', 'retained']].mean().round(2))

# grade distribution
print("\n=== Grade Distribution by Segment (%) ===")
print(pd.crosstab(dist_df['segment'], dist_df['path_level_grade'], normalize='index').round(3) * 100)

# payment frequency
print("\n=== Payment Frequency by Segment (%) ===")
print(pd.crosstab(dist_df['segment'], dist_df['payment_frequency_group'], normalize='index').round(3) * 100)

# free trial
print("\n=== Free Trial by Segment (%) ===")
print(pd.crosstab(dist_df['segment'], dist_df['free_trial'], normalize='index').round(3) * 100)

#gender
print("=== Gender by Segment (%) ===")
print(pd.crosstab(dist_df['segment'], dist_df['gender'], normalize='index').round(3) * 100)

#covid
print("\n=== Covid by Segment (%) ===")
print(pd.crosstab(dist_df['segment'], dist_df['covid'], normalize='index').round(3) * 100)

#device
print("\n=== device by Segment (%) ===")
print(pd.crosstab(dist_df['segment'], dist_df['device'], normalize='index').round(3) * 100)

=== Engagement by Segment ===
              total_time  total_completes  total_activities  retained
segment                                                              
Academic          132.38            28.24              8.12      0.73
Disengaged         88.38            19.61              7.40      0.64
Mixed             227.71            40.54             10.49      0.86
Recreational      180.99            34.20              9.59      0.80

=== Grade Distribution by Segment (%) ===
path_level_grade  First Grade  Kindergarten  PreK  Preschool  Second Grade  \
segment                                                                      
Academic                 11.0          25.3  21.9       20.7           7.6   
Disengaged                8.3          22.4  27.1       16.7           7.1   
Mixed                    15.1          30.8  29.8       10.5           9.1   
Recreational             15.1          29.9  26.2       15.3           7.1   

path_level_grade  Toddler Time  Unknow

Looking at the segment profiles, a few things stand out:

**Engagement:**
Mixed users are the most engaged. Averaging the highest time spent across both days (227 mins), 
completes (40), and retention (86%). Recreational users are 
second (180 mins, 80% retention). Academic users are 
surprisingly the least engaged of the three active segments 
(132 mins, 73% retention). Disengaged users have the lowest retention at 64%. Nearly 
a third of users who touched either content area didn't spend 
enough time to form a meaningful preference.

**Demographics:**
Grade distribution, payment frequency, and free trial rate 
are nearly identical across all segments. Academic and 
recreational users don't look like fundamentally different 
people — they just use the app differently.

**The counterintuitive finding:**
Toddler Time users are disproportionately represented in 
the Academic segment (11.4% vs 5.5% for Recreational) — 
suggesting parents of very young children may be driving 
academic content selection rather than the children themselves.
There could be something I could better understand to know why that is that isn't in the dataset (ex. Pet_park could be a social thing that appeals to slightly older kids, parents of toddlers actively seek out structured academic content)

### Does the data show that pet_park users are different from 
### mastery_math users?

### Short answer: behaviorally yes, demographically no.

**The PM's hypothesis is partially supported.**
The PM came with a hypothesis that two distinct user segmentations (academic and recreational) users existed and they had certain habits regarding pet_park and mastery_math. They were largely validated. The distribution of time share between pet_park and mastery_math 
shows a U-shaped bimodal pattern. 71% of users who engaged 
with either content area strongly lean toward one or the other. 
Two distinct usage patterns exist and the binary framing captures 
the majority of users well. 

However the binary framing misses meaningful nuance.
Using data-driven thresholds grounded in where users actually 
cluster (0.1 and 0.9), four natural segments emerge:
- Academic (26%) — predominantly mastery_math
- Recreational (16%) — predominantly pet_park
- Mixed (25%) — no strong preference
- Disengaged (31%) — touched one area but not meaningfully

**They behave differently but don't look different.**
Grade level, device, subscription type and free trial rate are 
nearly identical across segments. These aren't fundamentally 
different types of people — they just spend their time 
differently in the app.

**The most important finding for product:**
Mixed users — those who engage with both content areas — are 
the most engaged and retained segment (86% retention, 227 mins). 
Academic users are surprisingly the least retained active segment 
(73%). If retention is the goal, encouraging broad content 
exploration may be more valuable than doubling down on 
academic content alone.

If we wanted to go deeper on user segmentation beyond just pet_park and mastery_math, or if product didn't have a reasonable hypothesis on user segmentations, k-means clustering across all content areas could be a reasonable next step. It would let the data surface user types we potentially haven't anticipated rather than testing a specific hypothesis. I created an elbow plot below, but it shows no strong natural cluster structure. The curve declines gradually without a super clear bend. This suggests users exist more on a spectrum than in distinct behavioral buckets and the ratio-based segmentation we used is the most meaningful framework for these users.

In [25]:
content_cols = ['art', 'books', 'game', 'learn_music', 'puzzles',
                'song_chants', 'abcmtv', 'about_me', 'aquarium_hamster',
                'clsrm_subj', 'learn_at_home', 'library', 'mastery_math',
                'nav', 'pet_park', 'shopping', 'zoo_farm_theater']

engagement_cols = ['total_time', 'total_completes', 'total_activities']

feature_cols = content_cols + engagement_cols

content_df = df.groupby('userid')[content_cols].sum().reset_index()
kmeans_df = dist_df.merge(content_df, on='userid', how='inner')

X = kmeans_df[feature_cols].fillna(0)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

inertias = []
k_range = range(2, 11)

for k in k_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(X_scaled)
    inertias.append(kmeans.inertia_)

fig = px.line(
    x=list(k_range),
    y=inertias,
    markers=True,
    title='Elbow Plot — K-Means Across All Content Areas',
    labels={'x': 'Number of Clusters (k)', 'y': 'Inertia'}
)

fig.update_layout(
    plot_bgcolor='white',
    yaxis=dict(gridcolor='lightgrey'),
    xaxis=dict(gridcolor='lightgrey')
)

fig.show()

#  Question 3: Describe how you would set up an a/b test to assess whether doing “Mastery_Math” activities on Day0 contributes to a lift in the return rate on Day1.


First we look at 'is this a reasonable hypothesis?' As a data person, my first instinct is to ask why they have this hypothesis. Do people who use mastery math on day0 retain differently? The best a/b test is the one you don't even have to run. I can do some correlation analysis and likely see some signal that indicates whether or not this is a test worth allocating resources to



In [26]:
corr_df = con.execute("""
    WITH day0_users AS (
        SELECT
            userid,
            TRY_CAST(mastery_math AS DOUBLE) AS mastery_math_mins,
            TRY_CAST(tot_ts_mins AS DOUBLE) AS total_time,
            CASE WHEN TRY_CAST(mastery_math AS DOUBLE) >= 1 THEN 1 ELSE 0 END AS did_mastery_math
        FROM 'AoLAnalyticsTask.csv'
        WHERE which_day = 'day0'
    ),
    day1_retained AS (
        SELECT DISTINCT userid
        FROM 'AoLAnalyticsTask.csv'
        WHERE which_day = 'day1'
        AND n_act IS NOT NULL
    )
    SELECT
        d0.did_mastery_math,
        COUNT(*) AS total_users,
        SUM(CASE WHEN d1.userid IS NOT NULL THEN 1 ELSE 0 END) AS retained,
        ROUND(AVG(CASE WHEN d1.userid IS NOT NULL THEN 1.0 ELSE 0.0 END) * 100, 2) AS retention_rate,
        ROUND(AVG(d0.total_time), 2) AS avg_time_spent
    FROM day0_users d0
    LEFT JOIN day1_retained d1 ON d0.userid = d1.userid
    GROUP BY d0.did_mastery_math
""").df()

print(corr_df)

   did_mastery_math  total_users  retained  retention_rate  avg_time_spent
0                 0         2987    1768.0           59.19           48.13
1                 1         1371     938.0           68.42          100.59


First thing I notice is that users who do mastery_math do seem to retain better (68% vs 59%), but they're also spending more time (100 vs 48). We'll flag the confounding variable "avg time spent", because what we want to understand is 'is this retention increase real, or are we self-selecting for users that just like the product more?'

Below, I do some stratification. Rather than compare mastery math users to all non-mastery math users (because mastery math users are inherently more likely to like the platform as a whole), instead let's compare them within the same time usage buckets. I only want to compare users who spent similar amounts of time. For example, users who spent 30-60 mins on the product and DID mastery math vs users who spent 30-60 mins on the product and DIDN't do mastery math. If mastery math usage has real impact on retention, we'd expect to see a consistent retention advantage within each time band. If there isn't a difference in retention within bands, the overall retention change we saw above was unlikely to be caused by mastery math usage

In [27]:
strat_df = con.execute("""
    WITH day0_users AS (
        SELECT
            userid,
            TRY_CAST(mastery_math AS DOUBLE) AS mastery_math_mins,
            TRY_CAST(tot_ts_mins AS DOUBLE) AS total_time,
            CASE WHEN TRY_CAST(mastery_math AS DOUBLE) >= 1 THEN 1 ELSE 0 END AS did_mastery_math,
            CASE 
                WHEN TRY_CAST(tot_ts_mins AS DOUBLE) < 30 THEN '0-30 mins'
                WHEN TRY_CAST(tot_ts_mins AS DOUBLE) < 60 THEN '30-60 mins'
                WHEN TRY_CAST(tot_ts_mins AS DOUBLE) < 120 THEN '60-120 mins'
                ELSE '120+ mins'
            END AS time_band
        FROM 'AoLAnalyticsTask.csv'
        WHERE which_day = 'day0'
        AND n_act IS NOT NULL
    ),
    day1_retained AS (
        SELECT DISTINCT userid
        FROM 'AoLAnalyticsTask.csv'
        WHERE which_day = 'day1'
        AND n_act IS NOT NULL
    )
    SELECT
        d0.time_band,
        d0.did_mastery_math,
        COUNT(*) AS total_users,
        SUM(CASE WHEN d1.userid IS NOT NULL THEN 1 ELSE 0 END) AS retained,
        ROUND(AVG(CASE WHEN d1.userid IS NOT NULL THEN 1.0 ELSE 0.0 END) * 100, 2) AS retention_rate
    FROM day0_users d0
    LEFT JOIN day1_retained d1 ON d0.userid = d1.userid
    GROUP BY d0.time_band, d0.did_mastery_math
    ORDER BY d0.time_band, d0.did_mastery_math
""").df()

print(strat_df)

     time_band  did_mastery_math  total_users  retained  retention_rate
0    0-30 mins                 0         1434     683.0           47.63
1    0-30 mins                 1          222     102.0           45.95
2    120+ mins                 0          263     229.0           87.07
3    120+ mins                 1          373     327.0           87.67
4   30-60 mins                 0          728     446.0           61.26
5   30-60 mins                 1          332     191.0           57.53
6  60-120 mins                 0          562     410.0           72.95
7  60-120 mins                 1          444     318.0           71.62


In [28]:
strat_df['did_mastery_math'] = strat_df['did_mastery_math'].map({0: 'No Mastery Math', 1: 'Did Mastery Math'})

fig = px.bar(
    strat_df,
    x='time_band',
    y='retention_rate',
    color='did_mastery_math',
    barmode='group',
    title='Day1 Retention Rate by Time Band and Mastery Math Usage',
    labels={
        'time_band': 'Time Band',
        'retention_rate': 'Retention Rate (%)',
        'did_mastery_math': 'Mastery Math'
    },
    text='retention_rate',
    category_orders={'time_band': ['0-30 mins', '30-60 mins', '60-120 mins', '120+ mins']}
)

fig.update_traces(
    texttemplate='%{text}%',
    textposition='outside'
)

fig.update_layout(
    plot_bgcolor='white',
    yaxis=dict(gridcolor='lightgrey'),
    yaxis_range=[0, 110]
)

fig.show()

Looking at my results above, it looks like as we control for time spent on the platform, **there doesn't seem to be evidence that suggests those who did mastery math were more retained than those who don't**. Retention advantage we observed initially appears to be driven by overall engagement level, not mastery math usage itself. The hypothesis that mastery math drives day1 retention is not well supported by the data, and this test may not be worth running. But just to confirm, I'll fit a logistic regression below to confirm. 

Logistic regression below to confirm - control for time spent to isolate the independent effect of mastery math usage on retention. The model confirms that controlling for time spent, mastery math essentially has no independent effect on day1 retention

Why'd I also build the regression model?
1. Uses all users simultaneously rather than splitting into manual bands
2. Lets me answer "For every additional minute spent, here's how retention probability changes"

Ultimately, the stratification and the logistic regression tell the same story. Logistic regression is appropriate here because the outcome is binary. Is a user retained or not?

In [29]:
# build the dataset
log_df = con.execute("""
    WITH day0_users AS (
        SELECT
            userid,
            TRY_CAST(mastery_math AS DOUBLE) AS mastery_math_mins,
            TRY_CAST(tot_ts_mins AS DOUBLE) AS total_time,
            CASE WHEN TRY_CAST(mastery_math AS DOUBLE) >= 1 THEN 1 ELSE 0 END AS did_mastery_math
        FROM 'AoLAnalyticsTask.csv'
        WHERE which_day = 'day0'
        AND n_act IS NOT NULL
    ),
    day1_retained AS (
        SELECT DISTINCT userid
        FROM 'AoLAnalyticsTask.csv'
        WHERE which_day = 'day1'
        AND n_act IS NOT NULL
    )
    SELECT
        d0.userid,
        d0.did_mastery_math,
        d0.total_time,
        CASE WHEN d1.userid IS NOT NULL THEN 1 ELSE 0 END AS retained
    FROM day0_users d0
    LEFT JOIN day1_retained d1 ON d0.userid = d1.userid
""").df()

# features and target
X = log_df[['did_mastery_math', 'total_time']].fillna(0)
y = log_df['retained']

# scale
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# fit model
model = LogisticRegression()
model.fit(X_scaled, y)

# results
for feature, coef in zip(['did_mastery_math', 'total_time'], model.coef_[0]):
    print(f"{feature}: coefficient={round(coef, 4)}")

print(f"\nModel accuracy: {round(model.score(X_scaled, y) * 100, 2)}%")

did_mastery_math: coefficient=-0.0454
total_time: coefficient=0.9042

Model accuracy: 64.02%


A positive coefficient means that variable increases the probability 
of retention. A coefficient near zero means no meaningful effect. Since the total_time coefficient is high, this just basically means that the more time users spent on day0 correlated with returning on day1. Time spent is a positive predictor of retention. 

However, did_mastery_math has a coefficient of -0.0454. After controlling for time spent, doing mastery math actually is very slightly negatively associated with retention, but the effect is so close to zero that it's basically meaningless. 
**I'd tell product this means there's essentially no effect**

Correlation data suggests that this A/B likely isn't worth running. The retention difference you're seeing is likely explained by platform engagement, not mastery math specifically. I'd want to align on whether this changes our confidence in the hypothesis before committing to a test

## HOWEVER
If this data looked slightly different or we wanted to run the test anyway, here's what I would suggest:

**Effect size is king in A/B tests!** 
As the data person, my bias is always toward maximizing the separation between groups - the bigger the difference in mastery math exposure between test and control, the better our chance of detecting real signal (if one exists) is.

**Test Design Options:**

**Option A**: Maximum Signal (A/B/C test) - 33/33/33 split for new users
This test should be scoped to new users only because existing users have established behavior patterns that would confound the results. But also - we are testing for day1 retention, so naturally this will be an experiment for new users and how they onboard

- Group A: mastery math is heavily emphasized content on day0, or basically the user is forced into it somehow. Think sending the user there directly after the tutorial, etc.
- Group B: mastery math is heavily DE-emphasized on day 0. It is basically impossible to find, or maybe totally unavailable
- Group C: control group, normal experience

This will maximize effect size and give the clearest possible read on whether mastery math drives retention, but the tradeoff is it's pretty disruptive to the product experience and also is likely higher engineering lift

**Option B**: - moderate nudge (what I would recommend) - 50/50 split for new users
- Group A (test group): is nudged TOWARD mastery math in some way, maybe it's prominently placed or recommended first
- Group B (control group): is nudged AWAY from mastery math, maybe it's slightly more hidden or placed near the bottom of a list

This version is more product and engineering friendly, but it has a smaller effect size. This means we'll need more sample to detect real signal, but it's a reasonable balance between learning whether mastery math usage drives retention and product disruption.

Running these tests is a balance between how aggressively we want to learn vs how disruptive to the experience we're ok with being. I'd present both options and work with product to come up with a solution for how much disruption we're willing to accept in exchange for signal.


I'd run a power analysis using our observed baseline retention rate to determine required sample size and test duration for each option. The four inputs we care about are:
- Power (80% - meaning we accept a 20% chance of missing a real effect aka false negative, type II error)
- Alpha (0.05, meaning we accept a 5% chance of a false positive, type I error)
- Baseline retention rate 
- Minimum detectable effect (the minimum lift in retention we'd want to see to justify rolling this change out to everyone)

These are the inputs for the sample size formula: 
n = 2 × (Z_α + Z_β)² × p(1-p) / Δ²

This formula helps us calculate required sample size per group. We can then divide by daily new user volume to get test duration. I'd also recommend running the test for at least one full week to control for day of week behavioral differences
